In [ ]:
import numpy as np
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# custom
from _config import get_prms
from _trainer import Trainer
# devisce
device = torch.device("cuda:9" if torch.cuda.is_available() else "cpu")
print(device)

seed = 1234
np.random.seed(seed)
torch.manual_seed(seed)

# problem settings : exponential, logistic, target cell-limited 
name = 'lorenz' # 'exp', 'log', 'lorenz'
meth_i = 1
dist_i = 3
methods = ['gp', 'deeponet', 'eidgm']
dist_list = ['uni', 'bi', 'tri']
dist_type = dist_list[dist_i-1] # 'uni', 'bi', 'tri'
get_plot = True

# hyperparameter settings (prms[0]:DE, prms[1]:hyperPINN, prms[2]:WGAN)
prms = get_prms(name, dist_type=dist_type) # DE / hyperPINN / WGAN settings
prms[1]['deeponet'] = (meth_i==2)
if get_plot:
    prms[2]['num_noised'] = 1000//dist_i # Insert the number of real paramaters from the each nodes with noise 
    prms[2]['num_gen_plot'] = (1000-1)*prms[2]['num_bins']
else:
    prms[2]['num_noised'] = 1000//dist_i # Insert the number of real paramaters from the each nodes with noise 
    prms[2]['num_gen_plot'] = (1000-1)*prms[2]['num_bins']
#prms[2]['num_gen_ratio'] = 1 # Insert the ratio : fake data/real data 
prms[2]['num_cut'] = 0 # Insert the number of censored time for each mode which is randomly determined via numpy seed.
#prms

In [ ]:
# get dataset : if you want real data 
real_data = None # None, 'Abeta40', 'Abeta42'
trainer = Trainer(name, prms, device=device)
modes_noisy, X_data, X_data_onehot = trainer.get_data_gan(real_data=real_data)

> model output : gp / deeponet / hyperpinn 

In [ ]:
if meth_i in [2,3]:
    # prepare models
    # load pre-trained hyperPINNs
    trainer.load_pinn()

    # load WGANs
    load_params = True # if you want to train WGAN from the begining, then this should be False.
    trainer.load_gan(load_params=load_params)

    # show WGAN result
    gens = trainer.plot_results_gan(modes_noisy, X_data, plot_traj=False, return_gen=True, get_scores=True)

    print(len(modes_noisy), len(gens))
else:
    # GET GP save files
    gens = torch.load('./save/GP/gp_'+trainer.name+str(trainer.num_modes)+'.pth')
    print(len(modes_noisy), len(gens), trainer.WD(gens, modes_noisy))

> plot

In [ ]:
trues = pd.DataFrame(np.concatenate([modes_noisy.numpy(), np.reshape(np.array([['true']*len(modes_noisy)]),(-1,1))],-1), columns=trainer.param_names+[' ']).astype({key:float for key in trainer.param_names})
fakes = pd.DataFrame(np.concatenate([gens.numpy(), np.reshape(np.array([['estimated']*len(gens)]),(-1,1))],-1), columns=trainer.param_names+[' ']).astype({key:float for key in trainer.param_names})
df = pd.concat([fakes, trues])

cc = ['magenta','blue','green']
tc = cc[dist_i-1]
sns.set(font_scale=3)
sns.set_style("ticks", {'axes.grid' : False})

c_mod = tc
num_bins = trainer.num_bins
tmin, tmax = trainer.tmin, trainer.tmax
x_bins = trainer.t_numpy
t = np.linspace(tmin,tmax,num_bins)
t_torch = torch.linspace(tmin,tmax,num_bins)
y_values = []
# true traj
for mod in modes_noisy:
    y_values.append(trainer.solution(torch.cat([t_torch.view(-1,1), mod.view(-1,trainer.num_p).repeat(num_bins,1)], -1)))
# fake traj
if trainer.sc:
    p_output_unsc = self.scale_p(gens, backward=True)
else:
    p_output_unsc = gens
t_pred = torch.linspace(trainer.tmin, trainer.tmax, 101).view(-1,1) # for trajectories
y_vals = []
for i, mode in enumerate(gens):  # Assuming A_list_ap is your list of parameter sets
    y_val = trainer.solution(torch.concat([t_pred]+[p*torch.ones_like(t_pred) for p in mode],-1))
    if trainer.sc:
        y_vals.append(trainer.scale_y(y_val))
    else:
        y_vals.append(y_val)
        
if dist_i==1:
    fs = 32
    ttl1 = True
    xlb1=False
    ylb1=False
if dist_i==2:
    fs = 32
    ttl1 = False
    xlb1=False
    ylb1=True
if dist_i==3:
    fs = 35
    ttl1 = False
    xlb1=True
    ylb1=False
bw = 0.1

In [ ]:
# true data plot 
fig, axs = plt.subplots(1, 1, figsize=(4.5+0.8*int(ylb1), 4.5+0.4*(int(xlb1)+int(ttl1))), dpi=300)
#fig, axs = plt.figure(figsize=(6, 4.5), dpi=300)
# Solve and plot trajectories
# plot traj
k = 0
for y_val in y_vals:
    cp1 = axs.plot(t_pred, y_val[:,k].numpy().reshape(-1), linewidth=0.3, color='orange', alpha=0.2)

for yval in y_values:
    cp2 = axs.scatter(t_torch, yval[:,k], color=tc, s=40, alpha=1, zorder=10000)

# Plot formatting
axs.set_xlim([-0.05, 1.05])
axs.set_ylim([-16, 16])
axs.set_yticks([-10.0, 0.0, 10.0])
axs.set_xticks([0, 0.5, 1])
#axs.set_yscale('log')
if xlb1:
    axs.set_xlabel('', fontsize=fs)  # Larger font size for formal papers
else:
    axs.set_xlabel('', fontsize=fs)  # Larger font size for formal papers
    axs.set_xticklabels([])
if ylb1:
    axs.set_ylabel('', fontsize=fs)  # Larger font size for formal papers
else:
    axs.set_ylabel('', fontsize=fs)  # Larger font size for formal papers
    axs.set_yticklabels([])
axs.tick_params(axis='both', which='major', labelsize=fs)
if ttl1:
    axs.set_title('X', fontsize=fs)  # Title for the plot
else:
    axs.set_title('', fontsize=fs)  # Title for the plot
#plt.title('Observation', fontsize=16, fontweight='bold')  # Title for the plot
# Use a tight layout to ensure everything fits without overlap
for axis in ['bottom','left']:
    axs.spines[axis].set_linewidth(4)
for axis in ['top','right']:
    axs.spines[axis].set_linewidth(0)
axs.yaxis.set_tick_params(width=3)
axs.xaxis.set_tick_params(width=3)
plt.tight_layout()
# Save the figure with high resolution for better clarity in the paper
plt.savefig('./eps/lorenz'+str(dist_i)+'_trajX.eps', format='eps', dpi=300)
# Show the plot
plt.show()

# true data plot 
ylb1=False
c_mod = tc
num_bins = trainer.num_bins
tmin, tmax = trainer.tmin, trainer.tmax
x_bins = trainer.t_numpy
t = np.linspace(tmin,tmax,num_bins)
t_torch = torch.linspace(tmin,tmax,num_bins)
fig, axs = plt.subplots(1, 1, figsize=(4.5+0.8*int(ylb1), 4.5+0.5*(int(xlb1)+int(ttl1))), dpi=300)
#fig, axs = plt.figure(figsize=(6, 4.5), dpi=300)
# Solve and plot trajectories
# plot traj
k = 1
for y_val in y_vals:
    cp1 = axs.plot(t_pred, y_val[:,k].numpy().reshape(-1), linewidth=0.3, color='orange', alpha=0.2)

for yval in y_values:
    cp2 = axs.scatter(t_torch, yval[:,k], color=tc, s=40, alpha=1, zorder=10000)

# Plot formatting
axs.set_xlim([-0.05, 1.05])
axs.set_ylim([-20, 20])
axs.set_yticks([-10.0, 0.0, 10.0])
axs.set_xticks([0, 0.5, 1])
#axs.set_yscale('log')
if xlb1:
    axs.set_xlabel('', fontsize=fs)  # Larger font size for formal papers
else:
    axs.set_xlabel('', fontsize=fs)  # Larger font size for formal papers
    axs.set_xticklabels([])
if ylb1:
    axs.set_ylabel('Population', fontsize=fs)  # Larger font size for formal papers
else:
    axs.set_ylabel('', fontsize=fs)  # Larger font size for formal papers
    axs.set_yticklabels([])
axs.tick_params(axis='both', which='major', labelsize=fs)
if ttl1:
    axs.set_title('Y', fontsize=fs)  # Title for the plot
else:
    axs.set_title('', fontsize=fs)  # Title for the plot
#plt.title('Observation', fontsize=16, fontweight='bold')  # Title for the plot
# Use a tight layout to ensure everything fits without overlap
for axis in ['bottom','left']:
    axs.spines[axis].set_linewidth(4)
for axis in ['top','right']:
    axs.spines[axis].set_linewidth(0)
axs.yaxis.set_tick_params(width=3)
axs.xaxis.set_tick_params(width=3)
plt.tight_layout()
# Save the figure with high resolution for better clarity in the paper
plt.savefig('./eps/lorenz'+str(dist_i)+'_trajY.eps', format='eps', dpi=300)
# Show the plot
plt.show()

# true data plot 
ylb1=False
c_mod = tc
num_bins = trainer.num_bins
tmin, tmax = trainer.tmin, trainer.tmax
x_bins = trainer.t_numpy
t = np.linspace(tmin,tmax,num_bins)
t_torch = torch.linspace(tmin,tmax,num_bins)
fig, axs = plt.subplots(1, 1, figsize=(4.5+0.8*int(ylb1), 4.5+0.5*(int(xlb1)+int(ttl1))), dpi=300)
#fig, axs = plt.figure(figsize=(6, 4.5), dpi=300)
# Solve and plot trajectories
# plot traj
k = 2
for y_val in y_vals:
    cp1 = axs.plot(t_pred, y_val[:,k].numpy().reshape(-1), linewidth=0.3, color='orange', alpha=0.2)

for yval in y_values:
    cp2 = axs.scatter(t_torch, yval[:,k], color=tc, s=40, alpha=1, zorder=10000)

# Plot formatting
axs.set_xlim([-0.05, 1.05])
axs.set_ylim([5, 45])
axs.set_yticks([10, 25, 40])
axs.set_xticks([0, 0.5, 1])
#axs.set_yscale('log')
if xlb1:
    axs.set_xlabel('', fontsize=fs)  # Larger font size for formal papers
else:
    axs.set_xlabel('', fontsize=fs)  # Larger font size for formal papers
    axs.set_xticklabels([])
if ylb1:
    axs.set_ylabel('Population', fontsize=fs)  # Larger font size for formal papers
else:
    axs.set_ylabel('', fontsize=fs)  # Larger font size for formal papers
    axs.set_yticklabels([])
axs.tick_params(axis='both', which='major', labelsize=fs)
if ttl1:
    axs.set_title('Z', fontsize=fs)  # Title for the plot
else:
    axs.set_title('', fontsize=fs)  # Title for the plot
#plt.title('Observation', fontsize=16, fontweight='bold')  # Title for the plot
# Use a tight layout to ensure everything fits without overlap
for axis in ['bottom','left']:
    axs.spines[axis].set_linewidth(4)
for axis in ['top','right']:
    axs.spines[axis].set_linewidth(0)
axs.yaxis.set_tick_params(width=3)
axs.xaxis.set_tick_params(width=3)
plt.tight_layout()
# Save the figure with high resolution for better clarity in the paper
plt.savefig('./eps/lorenz'+str(dist_i)+'_trajZ.eps', format='eps', dpi=300)
# Show the plot
plt.show()

In [ ]:
fs = 27
if dist_i == 1:
    yl = True
    xlb1 = False
if dist_i == 2:
    yl = False
    xlb1 = False
if dist_i == 3:
    yl = False
    xlb1 = True

In [ ]:
bw = 0.1
plt.figure(figsize=(4,4), dpi=300)
g = sns.distplot(df.loc[df[' '] == 'true', 'sigma'], hist=False, rug=False, color=tc, kde_kws={'linewidth':5, 'fill':True, 'bw':bw})
g = sns.distplot(df.loc[df[' '] == 'estimated', 'sigma'], hist=False, rug=False, color='orange', kde_kws={'linewidth':5, 'fill':True, 'bw':bw})
if xlb1:
    g.set(ylabel='', xlabel=r'$\sigma$', xticks=[9, 10, 11], xlim=[7,13], yticks=[])
else:
    if yl:
        g.set(ylabel='', xlabel='', xticks=[9, 10, 11], xlim=[7,13], xticklabels=[], yticks=[])
    else:
        g.set(ylabel='', xlabel='', xticks=[9, 10, 11], xlim=[7,13], xticklabels=[], yticks=[])
for axis in ['bottom','left']:
    g.spines[axis].set_linewidth(4)
for axis in ['top','right']:
    g.spines[axis].set_linewidth(0)
g.xaxis.set_tick_params(width=3)
plt.savefig('./eps/lorenz'+str(dist_i)+'_'+methods[meth_i-1]+'_fake_sigma.eps', format='eps', dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(4,4), dpi=300)
g = sns.distplot(df.loc[df[' '] == 'true', 'rho'], hist=False, rug=False, color=tc, kde_kws={'linewidth':5, 'fill':True, 'bw':bw})
g = sns.distplot(df.loc[df[' '] == 'estimated', 'rho'], hist=False, rug=False, color='orange', kde_kws={'linewidth':5, 'fill':True, 'bw':bw})
if xlb1:
    g.set(ylabel='', xlabel=r'$\rho$', yticks=[], xticks=[18, 23, 28], xlim=[15,31])
else:
    if yl:
        g.set(ylabel='', xlabel='', yticks=[], xticks=[18, 23, 28], xlim=[15,31], xticklabels=[])
    else:
        g.set(ylabel='', xlabel='', yticks=[], xticks=[18, 23, 28], xlim=[15,31], xticklabels=[])
for axis in ['bottom','left']:
    g.spines[axis].set_linewidth(4)
for axis in ['top','right']:
    g.spines[axis].set_linewidth(0)
g.xaxis.set_tick_params(width=3)
plt.savefig('./eps/lorenz'+str(dist_i)+'_'+methods[meth_i-1]+'_fake_rho.eps', format='eps', dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(4,4), dpi=300)
g = sns.distplot(df.loc[df[' '] == 'true', 'beta'], hist=False, rug=False, color=tc, kde_kws={'linewidth':5, 'fill':True, 'bw':bw})
g = sns.distplot(df.loc[df[' '] == 'estimated', 'beta'], hist=False, rug=False, color='orange', kde_kws={'linewidth':5, 'fill':True, 'bw':bw})
if xlb1:
    g.set(ylabel='', xlabel=r'$\beta$', yticks=[], xticks=[1, 1.7, 2.4], xlim=[0.4,3])
else:
    if yl:
        g.set(ylabel='', xlabel='', xticks=[1, 1.7, 2.4], xlim=[0.4,3], xticklabels=[], yticks=[])
    else:
        g.set(ylabel='', xlabel='', xticks=[1, 1.7, 2.4], xlim=[0.4,3], xticklabels=[], yticks=[])
for axis in ['bottom','left']:
    g.spines[axis].set_linewidth(4)
for axis in ['top','right']:
    g.spines[axis].set_linewidth(0)
g.xaxis.set_tick_params(width=3)
plt.savefig('./eps/lorenz'+str(dist_i)+'_'+methods[meth_i-1]+'_fake_beta.eps', format='eps', dpi=300)
plt.show()